# TLS-Layer Characterization


This notebook helps you to:

1. Load CSV with characterization of web agent across TLS, IP and Browser layers.
2. Display figures and precise number not included in the paper.
3. Reproduice table 5 of section 6.2.
4. Reproduice tables 9 and 10 in appendix H.

In [1]:
import pandas as pd
import numpy as np

# Better dataframe display
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [ ]:
df = pd.read_csv("./active_tests_with_TLS_IP_layers.csv")
df_fps = pd.read_csv("./active_tests_browser_layers.csv")

df["Web Agent"] = df["Web Agent"].str.strip()
df["Agent Version"] = df["Agent Version"].astype(str).str.strip()

df = df.merge(
    df_fps[['_id', 'userAgentData_fullVersionList', 'userAgentData_brands']], 
    on='_id',
    how='left' 
)

## Table 5

This cell reproduces Table 5:

| Tool | JA4 All | JA4 Unique | Dominant JA4 | Dominant Intra. | Dominant V. Score |  JA4 A. Score |
| ---- | ------: | ---------: | ------------ | --------------: | ----------------: |  -----------: |

Definitions:

* **JA4 All** = all JA4 fingerprints observed for the tool.

* **JA4 Unique** = number of JA4 fingerprints observed only for the tool.

* **Dominant JA4** = most frequently observed JA4 fingerprint for the tool.

* **Dominant Intra.** = proportion of observations corresponding to the dominant JA4 fingerprint.

* **Dominant V. Score** = discriminative score of the dominant JA4 fingerprint, computed as: V-Score = Intra-Score × Inter-Score

* **JA4 A. Score** = overall discriminative score of the tool, computed as the sum of the V-Scores of all its JA4 fingerprints.



In [3]:
# ============================================================
# Helper functions to build the "Browser_Ver_extracted" column
# and the final "coarse_group" column used for grouping in the
# analysis for Table 5
# ============================================================
def parse_ua_string(ua_string):
    parsed = []
    parts = ua_string.split('|')

    for p in parts:
        if ':' in p:
            brand, version = p.split(':', 1)
            parsed.append({
                'brand': brand.strip(),
                'version': version.strip()
            })
    return parsed

# ===========================
# Helper extract browser full
# ===========================

def extract_browser_full(row):
    ua_full = row.get('userAgentData_fullVersionList')
    ua_brands = row.get('userAgentData_brands')

    agent_browser = str(row.get('Agent Browser', '')).lower()
    web_agent = str(row.get('Web Agent', '')).lower()
    fallback = row.get('Browser version')
    ja4 = row.get('ja4')

    # =========================
    # Special Case Agent Browser
    # =========================
    special_keywords = ['firefox', 'curl', 'wget', 'scrapy']
    if any(k in agent_browser for k in special_keywords):
        return row.get('Agent Browser')
    
    if web_agent == 'openclaw' and ja4 == 't13d3012h2_1d37bd780c83_882d495ac381':
        return "Python-Requests ???" # We know it's Python Requests, but the version is uncertain due to the JA4 hash

    result = None

    # =========================
    # PARSE userAgentData
    # =========================
    parsed = None
    
    if isinstance(ua_full, str) and ua_full.strip() not in ["", "Not supported", "MISSING"]:
        try:
            parsed = parse_ua_string(ua_full)
        except:
            parsed = None

    elif isinstance(ua_brands, str) and ua_brands.strip() not in ["", "Not supported", "MISSING"]:
        try:
            parsed = parse_ua_string(ua_brands)
        except:
            parsed = None

    # =========================
    # EXTRACTION STANDARD
    # =========================
    if parsed is not None:

         # --- Chrome / Edge
        for b in parsed:
            if b['brand'] in ['Google Chrome', 'Microsoft Edge']:
                result = f"{b['brand']} {b['version']}"
                break

        # --- Chromium fallback
        if result is None:
            for b in parsed:
                if b['brand'] == 'Chromium':
                    result = f"Chromium {b['version']}"
                    break

        # --- ChatGPT fallback
        if result is None and 'chatgpt' in web_agent:
            for b in parsed:
                if b['brand'] == 'Chromium':
                    result = f"Chromium {b['version']}"
                    break

    # =========================
    # Skyvern (fallback)
    # =========================
    if result is None and web_agent == 'skyvern':
        return "Microsoft Edge 144.0.3719.92"

    # =========================
    # Other cases
    # =========================
    if result is None:

        # --- playwright / crawl4AI
        if any(k in web_agent for k in ['playwright', 'crawl4ai']):
            return "Chromium 145.0.7632.6"

        # --- selenium / openclaw / anthropic claude
        if any(k in web_agent for k in ['selenium', 'openclaw', 'claude']):
            return "Google Chrome 144.0.7559.96"

        # --- puppeteer
        if any(k in web_agent for k in ['puppeteer']):
            return "Chromium 145.0.7632.46"
        
        # --- browserUse
        if 'browseruse' in web_agent:
            agent_val = row.get('Agent Browser')

            if agent_val == "Google Chrome 144.0.0.0":
                return "Google Chrome 144.0.7559.0" # We can only be sure about the major version, so we set minor/patch to 0
            
            return agent_val
        
        # --- chatgpt agent
        if 'chatgpt' in web_agent:
            return "Chromium 141"
        
        # --- human using chrome
        if 'human' in web_agent:
            return row.get('Agent Browser')

    # =========================
    # Last fallback
    # =========================
    if result is None:
        return fallback

    return result
    
df['Browser_Ver_extracted'] = df.apply(extract_browser_full, axis=1)

browser_col = df["Agent Browser"].fillna("").str.lower()

df["Browser_Family"] = np.select(
    [
        browser_col.str.contains(r"firefox"),
        browser_col.str.contains(r"edg|edge"),
        browser_col.str.contains(r"chromium"),
        browser_col.str.contains(r"chrome")
    ],
    [
        "Firefox",
        "Edge",
        "Chromium",
        "Chrome"
    ],
    default="Other"
)

mask = (
    (df["Web Agent"] == "Puppeteer") &
    (df["Browser_Family"] == "Firefox") &
    (df["Browser Local/Cloud"] == "LOCAL") &
    (df["Agent Version"].isin(["24.37.8", "24.37.9"]))
)

df.loc[mask, "Agent Version"] = "24.37.2"

df[
    (df["Web Agent"] == "Puppeteer") &
    (df["Browser_Family"] == "Firefox")
][["Agent Version"]].drop_duplicates()

group_cols = [
    "Web Agent", "Model", "Browser Local/Cloud",
    "Agent Version", "Browser_Ver_extracted"
]

# -----------------------------
# Define COARSE grouping (for A-score only)
# -----------------------------
def assign_coarse_group(row):
    agent = str(row["Web Agent"]).lower()
    env = str(row["Browser Local/Cloud"]).lower()
    browser = str(row["Browser_Ver_extracted"]).lower()

    # --- BrowserUse ---
    if "browseruse" in agent:
        if "stealth" in agent:
            return "BrowserUse-Stealth"
        elif "cloud" in env:
            return "BrowserUse-Cloud"
        else:
            return "BrowserUse-Local"

    # --- OpenClaw ---
    if agent == "openclaw" and not ("chrome" in browser):
        return "OpenClaw-Python-Request"

    elif agent == "openclaw":
        return "OpenClaw"
    
    # --- Claude ---
    if agent == "antropic claude for chrome":
        return "Antropic Claude for Chrome"

    # --- ChatGPT ---
    if agent == "chatgpt agent":
        return "ChatGPT"

    # --- Skyvern ---
    if agent == "skyvern":
        return "Skyvern"

    # --- fallback: keep original fine-grained identity ---
    return " | ".join([
        str(row["Web Agent"]),
        str(row["Model"]),
        str(row["Browser Local/Cloud"]),
        str(row["Agent Version"]),
        str(row["Browser_Ver_extracted"])
    ])

# Apply coarse grouping
df["coarse_group"] = df.apply(assign_coarse_group, axis=1)

In [4]:
# -----------------------------
# Intra: P(A = v | t)
# -----------------------------
ja4_prop = pd.crosstab(
    [df[c] for c in group_cols],
    df["ja4"],
    normalize="index"
)

# -----------------------------
# Fine counts (original t)
# -----------------------------
group_counts = pd.crosstab(
    [df[c] for c in group_cols],
    df["ja4"]
)

group_sizes = group_counts.sum(axis=1)
total_size = len(df)

# -----------------------------
# Global distribution
# -----------------------------
global_counts = df["ja4"].value_counts()

# -----------------------------
# Coarse counts
# -----------------------------
coarse_counts = pd.crosstab(
    df["coarse_group"],
    df["ja4"]
)

coarse_sizes = coarse_counts.sum(axis=1)

# -----------------------------
# Compute Inter with coarse groups
# -----------------------------
inter = pd.DataFrame(index=group_counts.index, columns=group_counts.columns)

for t in group_counts.index:
    # reconstruct row as dict
    row_dict = dict(zip(group_cols, t))
    coarse_t = assign_coarse_group(row_dict)

    for v in group_counts.columns:
        total_v = global_counts[v]

        count_v_coarse = coarse_counts.get(v, pd.Series()).get(coarse_t, 0)
        size_coarse = coarse_sizes.get(coarse_t, 0)

        if total_size - size_coarse > 0:
            p_out = (total_v - count_v_coarse) / (total_size - size_coarse)
        else:
            p_out = 0

        inter.loc[t, v] = 1 - p_out

inter = inter.astype(float)

# -----------------------------
# V-score and A-score
# -----------------------------
v_score = ja4_prop * inter
a_score = v_score.sum(axis=1)

# -----------------------------
# Existing metrics
# -----------------------------
dominant_ja4 = ja4_prop.idxmax(axis=1)
dominant_share = ja4_prop.max(axis=1)

# Inter score of the dominant JA4
dominant_inter = pd.Series(
    [inter.loc[idx, dominant_ja4.loc[idx]] for idx in inter.index],
    index=inter.index
)
# Dominant JA4 V-score
v_score_dominant = dominant_share * dominant_inter

ja4_div = df.groupby(group_cols)["ja4"].nunique()

# -----------------------------
# Summary
# -----------------------------
summary = pd.DataFrame({
    "Dominant_JA4": dominant_ja4,
    "Dominant_Intra": dominant_share,
    "Dominant_Inter": dominant_inter,
    "Dominant_V_Score": v_score_dominant,
    "JA4_Diversity": ja4_div,
    "JA4_A_Score": a_score
})

summary = summary.round(3)

# -----------------------------
# Rename JA4 (J1, J2, ...)
# -----------------------------
unique_vals = sorted(df["ja4"].unique())
unique_ja4 = {v: f"J{i+1}" for i, v in enumerate(unique_vals)}

summary["Dominant_JA4"] = summary["Dominant_JA4"].map(unique_ja4)

# -----------------------------
# Sort
# -----------------------------
summary = summary.sort_values("Dominant_Intra", ascending=False)

# -----------------------------
# Build COARSE summary (for display)
# -----------------------------

# Intra at coarse level
ja4_prop_coarse = pd.crosstab(
    df["coarse_group"],
    df["ja4"],
    normalize="index"
)

# Counts
coarse_counts = pd.crosstab(
    df["coarse_group"],
    df["ja4"]
)

coarse_sizes = coarse_counts.sum(axis=1)

# --- Inter (coarse vs rest) ---
total_size = len(df)
global_counts = df["ja4"].value_counts()

total_v_matrix = pd.DataFrame(
    [global_counts] * len(coarse_counts),
    index=coarse_counts.index
)

size_t_matrix = pd.DataFrame(
    [coarse_sizes] * len(coarse_counts.columns)
).T
size_t_matrix.columns = coarse_counts.columns

denominator = (total_size - size_t_matrix).replace(0, 1)

p_out = (total_v_matrix - coarse_counts) / denominator
inter_coarse = 1 - p_out

# --- Scores ---
v_score_coarse = ja4_prop_coarse * inter_coarse
a_score_coarse = v_score_coarse.sum(axis=1)

# --- Metrics ---
dominant_ja4_coarse = ja4_prop_coarse.idxmax(axis=1)
dominant_share_coarse = ja4_prop_coarse.max(axis=1)
# Inter score of the dominant JA4
dominant_inter_coarse = pd.Series(
    [inter_coarse.loc[idx, dominant_ja4_coarse.loc[idx]] for idx in inter_coarse.index],
    index=inter_coarse.index
)
# Dominant JA4 V-score at coarse level
v_score_coarse_dominant = dominant_share_coarse * dominant_inter_coarse

ja4_div_coarse = df.groupby("coarse_group")["ja4"].nunique()

# --- Final table ---
summary_coarse = pd.DataFrame({
    "Dominant_JA4": dominant_ja4_coarse,
    "Dominant_Intra": dominant_share_coarse,
    "Dominant_Inter": dominant_inter_coarse,
    "Dominant_V_Score": v_score_coarse_dominant,
    "JA4_Diversity": ja4_div_coarse,
    "JA4_A_Score": a_score_coarse
}).round(2)

# Rename JA4 labels
summary_coarse["Dominant_JA4"] = summary_coarse["Dominant_JA4"].map(unique_ja4)

# Sort
summary_coarse = summary_coarse.sort_values("Dominant_Intra", ascending=False)

# -----------------------------
# JA4 sets per coarse group
# -----------------------------

# All JA4 per group
ja4_sets = df.groupby("coarse_group")["ja4"].unique()

# Convert to sorted lists
ja4_sets = ja4_sets.apply(lambda x: sorted(x))

# -----------------------------
# Unique JA4 per group
# -----------------------------

# Global counts
global_counts = df["ja4"].value_counts()

def get_unique_ja4(group_name):
    vals = ja4_sets[group_name]
    unique_vals = []
    
    for v in vals:
        if global_counts[v] == coarse_counts.loc[group_name, v]:
            unique_vals.append(v)
    
    return sorted(unique_vals)

unique_ja4_sets = {
    g: get_unique_ja4(g)
    for g in ja4_sets.index
}

unique_ja4_sets = pd.Series(unique_ja4_sets)

# -----------------------------
# Map to J1, J2... format
# -----------------------------

def format_list(vals):
    return ", ".join(unique_ja4[v] for v in vals)

summary_coarse["JA4_All"] = ja4_sets.apply(format_list)
summary_coarse["JA4_Unique"] = unique_ja4_sets.apply(format_list)

# Reorder columns
summary_coarse = summary_coarse[
    ["JA4_All", "JA4_Unique", "Dominant_JA4", "Dominant_Intra", "Dominant_V_Score", "JA4_A_Score"]
]
summary_coarse

,JA4_All,JA4_Unique,Dominant_JA4,Dominant_Intra,Dominant_V_Score,JA4_A_Score
coarse_group,,,,,,
ChatGPT,J18,J18,J18,1.00,1.00,1.00
scrapy | NONE | LOCAL | 2.14.1 | Scrapy/2.14.1 (+https://scrapy.org),J19,J19,J19,1.00,1.00,1.00
OpenClaw-Python-Request,J20,,J20,1.00,1.00,1.00
BrowserUse-Stealth,"J10, J12",,J10,0.98,0.88,0.90
BrowserUse-Cloud,"J10, J12",,J10,0.98,0.89,0.91
Crawl4AI_Stealth | openai/gpt-4o-mini | LOCAL | 0.8.0 | Chromium 145.0.7632.6,"J1, J11",,J11,0.80,0.66,0.83
Selenium | NONE | LOCAL | 4.18.1 | Firefox 147.0.2,"J7, J16, J17",,J17,0.78,0.78,0.99
Crawl4AI_Undetected_Browser | openai/gpt-4o-mini | LOCAL | 0.8.0 | Chromium 145.0.7632.6,"J1, J11",,J11,0.78,0.64,0.82
Crawl4AI | openai/gpt-4o-mini | LOCAL | 0.8.0 | Chromium 145.0.7632.6,"J1, J3, J11",,J11,0.76,0.63,0.82


## Table 9

This cell reproduces Table 9:

| JA4 ID | JA4 Full Hash |
| ------ | ------------- |

Definitions:

* **JA4 ID** = short identifier used throughout the paper and analysis notebooks to refer to a JA4 fingerprint.
* **JA4 Full Hash** = complete JA4 fingerprint as extracted from the corresponding TLS or QUIC handshake.

In [5]:
# Sorted unique JA4 for stable numbering
ja4_sorted = sorted(df["ja4"].dropna().unique())

ja4_mapping = {full: f"J{i+1}" for i, full in enumerate(ja4_sorted)}

# Reverse mapping (for appendix table)
ja4_reverse_mapping = {v: k for k, v in ja4_mapping.items()}

ja4_lookup_table = (
    pd.DataFrame({
        "JA4_ID": list(ja4_reverse_mapping.keys()),
        "JA4_Full_Hash": list(ja4_reverse_mapping.values())
    })
    .sort_values(by="JA4_ID", key=lambda x: x.str[1:].astype(int))
)
ja4_lookup_table

,JA4_ID,JA4_Full_Hash
0,J1,q13d0311h3_55b375c5d22e_653d80c3fe9d
1,J2,q13d0312h3_55b375c5d22e_178839b6cec1
2,J3,q13d0312h3_55b375c5d22e_5a06198afb93
3,J4,q13d0313h3_55b375c5d22e_b0954bf1abdf
4,J5,q13d0314h3_55b375c5d22e_61e396c58b1f
5,J6,q13d0314h3_55b375c5d22e_9dd3975af409
6,J7,q13d0315h3_55b375c5d22e_dc5437974b47
7,J8,q13d0316h3_55b375c5d22e_dc4af083c550
8,J9,t13d1412h2_e33ad33b3d25_6b314db333b6
9,J10,t13d1516h2_8daaf6152771_02713d6af862


## Table 10

This cell reproduces Table 10:

| Web Agent, Model, Browser Local/Cloud, Agent Version, Browser Ver. Extracted | J1 | J2 | ... | J21 |
| ---------------------------------------------------------------------------- | -: | -: | --- | --: |

Definitions:

* **Web Agent** = evaluated Web Agent.
* **Model** = underlying LLM model used by the agent.
* **Browser Local/Cloud** = whether the browser is executed locally on the user's machine or remotely in cloud infrastructure.
* **Agent Version** = version of the evaluated Web Agent.
* **Browser Ver. Extracted** = browser version extracted from the agent during the experiments using browser fingerprinting techniques.
* **J1–J21** = number of observations associated with each JA4 fingerprint. The mapping between JA4 identifiers and full JA4 hashes is provided in Table 9.

In [6]:
ja4_counts = pd.crosstab(
    [df["Web Agent"], df["Model"], df["Browser Local/Cloud"],
     df["Agent Version"], df['Browser_Ver_extracted']],
    df["ja4"]
)

ja4_counts = ja4_counts.rename(columns=ja4_mapping)

ja4_counts = ja4_counts[
    sorted(ja4_counts.columns, key=lambda x: int(x[1:]))
]

ja4_counts

ja4                                                                                                                 J1  J2  J3  J4  J5  J6  J7  J8  J9  J10  J11  J12  J13  J14  J15  J16  J17  J18  \
Web Agent                  Model             Browser Local/Cloud Agent Version Browser_Ver_extracted                                                                                                  
Antropic Claude for Chrome claude-opus-4.5   LOCAL               1.0.40        Google Chrome 144.0.7559.96          25   0  40   0   0   0   0   0   0    0    2    0    0    0    0    0    0    0   
                           claude-sonnet-4.5 LOCAL               1.0.40        Google Chrome 144.0.7559.96          21   0  35   0   0   0   0   0   0    0    1    0    0    0    0    0    0    0   
BrowserUse                 bu-1-0            LOCAL               0.11.5        Google Chrome 144.0.7559.96          38   0  17   0   0   0   0   0   0    0   10    0    0    0    0    0    0    0   
                           bu-2-0            CLOUD               0.11.6        Google Chrome 144.0.7559.110          0   0   0   0   0   0   0   0   0    1    0    0    0    0    0    0    0    0   
                                                                               Google Chrome 144.0.7559.60           0   0   0   0   0   0   0   0   0    2    0    0    0    0    0    0    0    0   
...                                                                                                                 ..  ..  ..  ..  ..  ..  ..  ..  ..  ...  ...  ...  ...  ...  ...  ...  ...  ...   
Skyvern                    Skyvern Optimized CLOUD               1.0.10        Microsoft Edge 143.0.3650.139         0   0   0   0   0   0   0   0   0    0    1    0    3    0    0    0    0    0   
                                                                               Microsoft Edge 144.0.3719.92          0   0   0   0   0   0   0   0   0    0   74    0   61    0    0    0    0    0   
curl                       NONE              LOCAL               8.15.0-DEV    curl/8.15.0-DEV                       0   0   0   0   0   0   0   0   1    0    0    0    0    0    0    0    0    0   
scrapy                     NONE              LOCAL               2.14.1        Scrapy/2.14.1 (+https://scrapy.org)   0   0   0   0   0   0   0   0   0    0    0    0    0    0    0    0    0    0   
wget                       NONE              LOCAL               1.21.4        Wget/1.21.4                           0   0   0   0   0   0   0   0   1    0    0    0    0    0    0    0    0    0   

ja4                                                                                                                 J19  J20  J21  
Web Agent                  Model             Browser Local/Cloud Agent Version Browser_Ver_extracted                               
Antropic Claude for Chrome claude-opus-4.5   LOCAL               1.0.40        Google Chrome 144.0.7559.96            0    0    0  
                           claude-sonnet-4.5 LOCAL               1.0.40        Google Chrome 144.0.7559.96            0    0    0  
BrowserUse                 bu-1-0            LOCAL               0.11.5        Google Chrome 144.0.7559.96            0    0    0  
                           bu-2-0            CLOUD               0.11.6        Google Chrome 144.0.7559.110           0    0    0  
                                                                               Google Chrome 144.0.7559.60            0    0    0  
...                                                                                                                 ...  ...  ...  
Skyvern                    Skyvern Optimized CLOUD               1.0.10        Microsoft Edge 143.0.3650.139          0    0    0  
                                                                               Microsoft Edge 144.0.3719.92           0    0    0  
curl                       NONE              LOCAL               8.15.0-DEV    curl/8.15.0-D